In [1]:
pip install pyarrow 

Note: you may need to restart the kernel to use updated packages.


In [3]:
import duckdb as db
import pandas as pd
import pyarrow as py

In [5]:
import os

In [7]:
import zipfile 


In [19]:
base_path=r"D:\loan_data"
temp_path=r"C:\Users\snarb\OneDrive\Desktop\loan_pipeline\temp"

In [165]:
year=2013
Quarter=4


In [167]:
year_zip_path=f"{base_path}/historical_data_{year}.zip"
print("opening zip year....")
with zipfile.ZipFile(year_zip_path, 'r') as year_zip:
    quarter_zip_name=f"historical_data_{year}Q{Quarter}.zip"
    print("extracting quarter zip")
    year_zip.extract(quarter_zip_name, temp_path)
    quarter_zip_path=os.path.join(temp_path, quarter_zip_name)
    print("Opening quarter zip...")
    with zipfile.ZipFile(quarter_zip_path, 'r') as q_zip:
        files = q_zip.namelist()
        orig_file = None
        perf_file = None
        
        for file in files:
            if "time" in file.lower():
                perf_file = file
            else:
                orig_file = file
        
        print("Detected files:")
        print("Origination:", orig_file)
        print("Performance:", perf_file)

# Extract using detected names
        q_zip.extract(orig_file, temp_path)
        q_zip.extract(perf_file, temp_path)


print("DONE")

opening zip year....
extracting quarter zip
Opening quarter zip...
Detected files:
Origination: historical_data_2013Q4.txt
Performance: historical_data_time_2013Q4.txt
DONE


In [169]:
con=db.connect()

In [171]:
orig_path = os.path.join(temp_path, orig_file)
perf_path = os.path.join(temp_path, perf_file)
# Output paths
orig_out = f"C:/Users/snarb/OneDrive/Desktop/loan_pipeline/parquet/origination/year={year}"
perf_out = f"C:/Users/snarb/OneDrive/Desktop/loan_pipeline/parquet/performance/year={year}"

os.makedirs(orig_out, exist_ok=True)
os.makedirs(perf_out, exist_ok=True)

print("Converting origination file...")

con.execute(f"""
COPY (
    SELECT * FROM read_csv_auto('{orig_path}', delim='|', all_varchar=True)
) TO '{orig_out}/q={Quarter}.parquet' (FORMAT PARQUET);
""")
print("Conversion DONE")

Converting origination file...
Conversion DONE


In [173]:
print("Converting origination file...")

con.execute(f"""
COPY (
    SELECT * FROM read_csv_auto('{perf_path}', delim='|', all_varchar=True)
) TO '{perf_out}/q={Quarter}.parquet' (FORMAT PARQUET);
""")
print("Conversion DONE")

Converting origination file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Conversion DONE


In [75]:
result = con.execute("""
SELECT 
    column03 AS delinquency_status,
    COUNT(*) as count
FROM 'C:/Users/snarb/OneDrive/Desktop/loan_pipeline/parquet/performance/**/*.parquet'
GROUP BY column03
ORDER BY count DESC
""").fetchall()

print(result)

[('0', 34339683), ('1', 250927), ('2', 65033), ('3', 29925), ('RA', 24720), ('4', 20586), ('5', 17097), ('6', 14067), ('7', 11879), ('8', 10396), ('9', 9118), ('10', 8063), ('11', 7246), ('12', 6365), ('13', 5648), ('14', 5032), ('15', 4487), ('16', 3901), ('17', 3530), ('18', 3137), ('19', 2786), ('20', 2557), ('21', 2291), ('22', 2061), ('23', 1850), ('24', 1740), ('25', 1598), ('26', 1401), ('27', 1313), ('28', 1238), ('29', 1139), ('30', 1088), ('31', 1009), ('32', 930), ('33', 850), ('34', 782), ('35', 754), ('36', 700), ('37', 667), ('38', 630), ('39', 586), ('40', 557), ('41', 524), ('42', 483), ('43', 462), ('44', 428), ('45', 398), ('46', 374), ('47', 353), ('48', 330), ('49', 320), ('50', 282), ('51', 267), ('52', 249), ('53', 237), ('54', 224), ('55', 206), ('56', 188), ('58', 180), ('57', 173), ('59', 163), ('60', 144), ('61', 140), ('62', 129), ('63', 118), ('64', 113), ('65', 109), ('66', 101), ('67', 93), ('68', 91), ('69', 85), ('70', 80), ('71', 76), ('72', 70), ('74',

In [81]:
con.execute("""
CREATE OR REPLACE VIEW performance_clean AS
SELECT 
    column00  AS loan_id,

    -- Dates
    CAST(column01 AS INTEGER) AS reporting_period,

    -- Balances
    CAST(column02 AS DOUBLE) AS current_actual_upb,

    -- Target / behavior
    column03 AS delinquency_status,

    -- Time features
    CAST(column04 AS INTEGER) AS loan_age,
    CAST(column05 AS INTEGER) AS remaining_months_to_maturity,

    -- Events
    CAST(column06 AS INTEGER) AS defect_settlement_date,
    column07 AS modification_flag,
    column08 AS zero_balance_code,
    CAST(column09 AS INTEGER) AS zero_balance_effective_date,

    -- Rates
    CAST(column10 AS DOUBLE) AS current_interest_rate,

    -- UPB breakdown
    CAST(column11 AS DOUBLE) AS non_interest_bearing_upb,

    -- Payment tracking
    CAST(column12 AS INTEGER) AS ddlpi,

    -- Recoveries / Loss
    CAST(column13 AS DOUBLE) AS mi_recoveries,
    column14 AS net_sale_proceeds,
    CAST(column15 AS DOUBLE) AS non_mi_recoveries,

    -- Expenses
    CAST(column16 AS DOUBLE) AS total_expenses,
    CAST(column17 AS DOUBLE) AS legal_costs,
    CAST(column18 AS DOUBLE) AS maintenance_costs,
    CAST(column19 AS DOUBLE) AS taxes_insurance,
    CAST(column20 AS DOUBLE) AS misc_expenses,

    -- Loss metric
    CAST(column21 AS DOUBLE) AS actual_loss,

    -- Modification economics
    CAST(column22 AS DOUBLE) AS cumulative_modification_cost,
    column23 AS step_modification_flag,
    column24 AS payment_deferral_flag,

    -- Risk metrics
    CAST(column25 AS DOUBLE) AS estimated_ltv,

    -- Default mechanics
    CAST(column26 AS DOUBLE) AS zero_balance_removal_upb,
    CAST(column27 AS DOUBLE) AS delinquent_accrued_interest,

    -- Flags
    column28 AS disaster_flag,
    column29 AS borrower_assistance_status,

    -- Monthly mod cost
    CAST(column30 AS DOUBLE) AS current_month_mod_cost,

    -- Core balance
    CAST(column31 AS DOUBLE) AS interest_bearing_upb

FROM 'C:/Users/snarb/OneDrive/Desktop/loan_pipeline/parquet/performance/**/*.parquet';
""")

In [85]:
con.execute("""
CREATE OR REPLACE VIEW origination_clean AS
SELECT 
    column00  AS credit_score,
    column01  AS first_payment_date,
    column02  AS first_time_homebuyer_flag,
    column03  AS maturity_date,
    column04  AS msa,

    -- Insurance
    column05  AS mi_percentage,

    -- Property
    column06  AS num_units,
    column07  AS occupancy_status,

    -- Risk ratios
    column08  AS cltv,
    column09  AS dti,

    -- Loan size
    column10 AS original_upb,
    column11 AS ltv,

    -- Pricing
    column12 AS original_interest_rate,

    -- Origination channel
    column13 AS channel,

    -- Loan structure
    column14 AS ppm_flag,
    column15 AS amortization_type,

    -- Geography
    column16 AS property_state,
    column17 AS property_type,
    column18 AS postal_code,

    -- ID
    column19 AS loan_id,

    -- Purpose
    column20 AS loan_purpose,

    -- Term
    column21 AS original_loan_term,

    -- Borrower
    column22 AS num_borrowers,

    -- Institutions
    column23 AS seller_name,
    column24 AS servicer_name,

    -- Flags
    column25 AS super_conforming_flag,
    column26 AS pre_refinance_loan_id,
    column27 AS program_type,
    column28 AS relief_refinance_flag,

    -- Valuation
    column29 AS valuation_method,

    -- Structure
    column30 AS interest_only_flag,
    column31 AS mi_cancellation_flag

FROM 'C:/Users/snarb/OneDrive/Desktop/loan_pipeline/parquet/origination/**/*.parquet';
""")

In [91]:
con.execute("""
CREATE OR REPLACE VIEW origination_fixed AS
SELECT 
    loan_id,

    -- Numeric cleaning
    NULLIF(credit_score, 9999) AS credit_score,
    NULLIF(cltv, 999) AS cltv,
    NULLIF(dti, 999) AS dti,
    NULLIF(ltv, 999) AS ltv,

    -- Replace invalid categorical codes
    CASE WHEN first_time_homebuyer_flag = '9' THEN NULL ELSE first_time_homebuyer_flag END AS first_time_homebuyer_flag,
    CASE WHEN occupancy_status = '9' THEN NULL ELSE occupancy_status END AS occupancy_status,
    CASE WHEN property_type = '99' THEN NULL ELSE property_type END AS property_type,

    -- Keep rest as-is
    first_payment_date,
    maturity_date,
    msa,
    mi_percentage,
    num_units,
    original_upb,
    original_interest_rate,
    channel,
    ppm_flag,
    amortization_type,
    property_state,
    postal_code,
    loan_purpose,
    original_loan_term,
    num_borrowers,
    seller_name,
    servicer_name,
    super_conforming_flag,
    program_type,
    valuation_method,
    interest_only_flag,
    mi_cancellation_flag

FROM origination_clean;
""")

In [95]:
con.execute("""
CREATE OR REPLACE VIEW performance_fixed AS
SELECT 
    loan_id,
    reporting_period,

    current_actual_upb,

    -- Clean delinquency
    CASE 
        WHEN delinquency_status = 'RA' THEN 'RA'
        WHEN delinquency_status IN ('0','1','2','3','4','5','6','7','8','9','10')
            THEN delinquency_status
        ELSE NULL
    END AS delinquency_status,

    loan_age,
    remaining_months_to_maturity,

    -- Replace "U" and weird values
    NULLIF(net_sale_proceeds, 'U') AS net_sale_proceeds,

    -- Replace numeric fake nulls
    NULLIF(estimated_ltv, 999) AS estimated_ltv,

    actual_loss,

    zero_balance_code,
    zero_balance_effective_date

FROM performance_clean;
""")

In [97]:
con.execute("""
CREATE OR REPLACE VIEW performance_with_target AS
SELECT *,
    CASE 
        WHEN delinquency_status IN ('3','4','5','6','7','8','9','10') THEN 1
        WHEN delinquency_status = 'RA' THEN 1
        ELSE 0
    END AS default_flag
FROM performance_fixed;
""")

In [99]:
con.execute("""
CREATE OR REPLACE VIEW loan_master AS
SELECT 
    p.loan_id,
    p.reporting_period,

    -- Target
    p.default_flag,

    -- Time dynamics
    p.loan_age,
    p.remaining_months_to_maturity,
    p.current_actual_upb,

    -- Loss / terminal info
    p.zero_balance_code,
    p.actual_loss,

    -- Origination features
    o.credit_score,
    o.cltv,
    o.dti,
    o.ltv,
    o.original_upb,
    o.original_interest_rate,
    o.loan_purpose,
    o.property_type,
    o.occupancy_status,
    o.num_units,
    o.num_borrowers,
    o.channel

FROM performance_with_target p
JOIN origination_fixed o
ON p.loan_id = o.loan_id;
""")

In [105]:
con.execute("""
CREATE OR REPLACE VIEW performance_enhanced AS
SELECT 
    loan_id,
    reporting_period,

    current_actual_upb,

    -- Keep original
    delinquency_status,

    -- 🔥 TRUE DPD (NO BUCKETING)
    CASE 
        WHEN delinquency_status = 'RA' THEN 999
        ELSE CAST(delinquency_status AS INTEGER)
    END AS dpd_numeric,

    -- 🔥 Default flag (>=90 DPD OR RA)
    CASE 
        WHEN delinquency_status = 'RA' THEN 1
        WHEN CAST(delinquency_status AS INTEGER) >= 90 THEN 1
        ELSE 0
    END AS is_default_state,

    loan_age,
    remaining_months_to_maturity,
    actual_loss,
    zero_balance_code,
    zero_balance_effective_date

FROM performance_fixed

-- 🔥 KEEP ONLY VALID ROWS
WHERE delinquency_status IS NOT NULL
  AND (
        delinquency_status = 'RA'
        OR TRY_CAST(delinquency_status AS INTEGER) IS NOT NULL
      );
""")

In [107]:
con.execute("""
SELECT 
    delinquency_status,
    dpd_numeric,
    COUNT(*) as cnt
FROM performance_enhanced
GROUP BY 1,2
ORDER BY dpd_numeric;
""").fetchall()


[('0', 0, 34339683),
 ('1', 1, 250927),
 ('2', 2, 65033),
 ('3', 3, 29925),
 ('4', 4, 20586),
 ('5', 5, 17097),
 ('6', 6, 14067),
 ('7', 7, 11879),
 ('8', 8, 10396),
 ('9', 9, 9118),
 ('10', 10, 8063),
 ('RA', 999, 24720)]

In [115]:
con.execute("""
SELECT 
    MIN(dpd_numeric),
    MAX(dpd_numeric),
    COUNT(*)
FROM performance_enhanced;
""").fetchall()

[(0, 999, 34801494)]

In [117]:
con.execute("""
CREATE OR REPLACE VIEW performance_enhanced AS
SELECT 
    loan_id,
    reporting_period,

    current_actual_upb,

    delinquency_status,

    -- 🔥 RAW DPD
    CASE 
        WHEN delinquency_status = 'RA' THEN 999
        ELSE CAST(delinquency_status AS INTEGER)
    END AS dpd_numeric,

    -- 🔥 CAPPED DPD (FOR MODELING)
    CASE 
        WHEN delinquency_status = 'RA' THEN 180
        WHEN CAST(delinquency_status AS INTEGER) > 180 THEN 180
        ELSE CAST(delinquency_status AS INTEGER)
    END AS dpd_capped,

    -- Default flag
    CASE 
        WHEN delinquency_status = 'RA' THEN 1
        WHEN CAST(delinquency_status AS INTEGER) >= 90 THEN 1
        ELSE 0
    END AS is_default_state,

    loan_age,
    remaining_months_to_maturity,
    actual_loss,
    zero_balance_code,
    zero_balance_effective_date

FROM performance_fixed

WHERE delinquency_status IS NOT NULL
  AND (
        delinquency_status = 'RA'
        OR TRY_CAST(delinquency_status AS INTEGER) IS NOT NULL
      );
""")

In [119]:
con.execute("""
SELECT 
    MIN(dpd_numeric),
    MAX(dpd_numeric),
    MIN(dpd_capped),
    MAX(dpd_capped)
FROM performance_enhanced;
""").fetchall()

[(0, 999, 0, 180)]

In [121]:
con.execute("""
CREATE OR REPLACE VIEW performance_lagged AS
SELECT 

    loan_id,
    reporting_period,

    -- Core behavior
    dpd_capped,
    is_default_state,

    -- 🔥 LAG FEATURES (MEMORY)
    LAG(dpd_capped, 1) OVER w AS dpd_lag_1,
    LAG(dpd_capped, 3) OVER w AS dpd_lag_3,
    LAG(dpd_capped, 6) OVER w AS dpd_lag_6,

    -- Default memory
    LAG(is_default_state, 1) OVER w AS default_lag_1,

    -- Carry forward useful vars
    current_actual_upb,
    loan_age,
    remaining_months_to_maturity

FROM performance_enhanced

WINDOW w AS (
    PARTITION BY loan_id 
    ORDER BY reporting_period
);
""")

In [123]:
con.execute("""
SELECT 
    loan_id,
    reporting_period,
    dpd_capped,
    dpd_lag_1,
    dpd_lag_3,
    dpd_lag_6
FROM performance_lagged
WHERE loan_id = (
    SELECT loan_id FROM performance_lagged LIMIT 1
)
ORDER BY reporting_period
LIMIT 20;
""").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('F09Q10168796', 200902, 0, None, None, None),
 ('F09Q10168796', 200903, 0, 0, None, None),
 ('F09Q10168796', 200904, 0, 0, None, None),
 ('F09Q10168796', 200905, 0, 0, 0, None),
 ('F09Q10168796', 200906, 0, 0, 0, None),
 ('F09Q10168796', 200907, 0, 0, 0, None),
 ('F09Q10168796', 200908, 0, 0, 0, 0),
 ('F09Q10168796', 200909, 0, 0, 0, 0),
 ('F09Q10168796', 200910, 0, 0, 0, 0),
 ('F09Q10168796', 200911, 0, 0, 0, 0),
 ('F09Q10168796', 200912, 0, 0, 0, 0),
 ('F09Q10168796', 201001, 0, 0, 0, 0),
 ('F09Q10168796', 201002, 0, 0, 0, 0),
 ('F09Q10168796', 201003, 0, 0, 0, 0),
 ('F09Q10168796', 201004, 0, 0, 0, 0),
 ('F09Q10168796', 201005, 0, 0, 0, 0),
 ('F09Q10168796', 201006, 0, 0, 0, 0),
 ('F09Q10168796', 201007, 0, 0, 0, 0),
 ('F09Q10168796', 201008, 0, 0, 0, 0),
 ('F09Q10168796', 201009, 0, 0, 0, 0)]

In [125]:
con.execute("""
CREATE OR REPLACE VIEW performance_features AS
SELECT 

    loan_id,
    reporting_period,

    dpd_capped,
    is_default_state,

    -- Lags (carry forward)
    dpd_lag_1,
    dpd_lag_3,
    dpd_lag_6,

    -- 🔥 ROLLING FEATURES

    -- Avg DPD (last 3 months)
    AVG(dpd_capped) OVER w3 AS avg_dpd_3m,

    -- Max DPD (last 6 months)
    MAX(dpd_capped) OVER w6 AS max_dpd_6m,

    -- Count of delinquency (>=30 DPD) in last 6 months
    SUM(CASE WHEN dpd_capped >= 30 THEN 1 ELSE 0 END) OVER w6 AS count_dpd_30_6m,

    -- 🔥 TREND FEATURE (simple slope proxy)
    (dpd_capped - dpd_lag_3) AS dpd_trend_3m,

    -- Carry useful vars
    current_actual_upb,
    loan_age,
    remaining_months_to_maturity

FROM performance_lagged

-- Windows
WINDOW 
    w3 AS (
        PARTITION BY loan_id 
        ORDER BY reporting_period 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),
    
    w6 AS (
        PARTITION BY loan_id 
        ORDER BY reporting_period 
        ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
    );
""")

In [127]:
con.execute("""
SELECT 
    dpd_capped,
    avg_dpd_3m,
    max_dpd_6m,
    count_dpd_30_6m,
    dpd_trend_3m
FROM performance_features
WHERE loan_id = (
    SELECT loan_id FROM performance_features LIMIT 1
)
ORDER BY reporting_period
LIMIT 20;
""").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(0, 0.0, 0, 0, None),
 (0, 0.0, 0, 0, None),
 (0, 0.0, 0, 0, None),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0),
 (0, 0.0, 0, 0, 0)]

In [133]:
con.execute("""
SELECT 
    loan_id,
    MAX(dpd_capped) as max_dpd
FROM performance_features
GROUP BY loan_id
HAVING MAX(dpd_capped) >= 90
LIMIT 5;
""").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('F09Q10073224', 180),
 ('F09Q10091736', 180),
 ('F09Q10104067', 180),
 ('F09Q10127942', 180),
 ('F09Q10591147', 180)]

In [157]:
con.execute("""
CREATE OR REPLACE VIEW performance_final AS
SELECT 

    loan_id,
    reporting_period,

    dpd_capped,
    is_default_state,

    -- Carry previous features
    dpd_lag_1,
    dpd_lag_3,
    dpd_lag_6,
    avg_dpd_3m,
    max_dpd_6m,
    count_dpd_30_6m,
    dpd_trend_3m,

    -- 🔥 TRANSITION FEATURES

    -- Change
    (dpd_capped - dpd_lag_1) AS dpd_change_1m,

    CASE 
        WHEN dpd_capped > dpd_lag_1 THEN 1 ELSE 0
    END AS is_worsening,

    CASE 
        WHEN dpd_capped < dpd_lag_1 THEN 1 ELSE 0
    END AS is_improving,

    -- Roll forward (only if already delinquent)
    CASE 
        WHEN dpd_lag_1 >= 30 AND dpd_capped > dpd_lag_1 THEN 1 ELSE 0
    END AS roll_forward_flag,

    -- Cure
    CASE 
        WHEN dpd_lag_1 >= 30 AND dpd_capped < dpd_lag_1 THEN 1 ELSE 0
    END AS cure_flag,

    -- 🔥 PERSISTENCE FEATURES

    -- Consecutive delinquency counter
    CASE 
        WHEN dpd_capped >= 30 THEN 
            ROW_NUMBER() OVER (
                PARTITION BY loan_id, 
                CASE WHEN dpd_capped >= 30 THEN 1 ELSE 0 END
                ORDER BY reporting_period
            )
        ELSE 0
    END AS consecutive_delinquency_count,

    -- Months since last current (approx)
    SUM(CASE WHEN dpd_capped = 0 THEN 1 ELSE 0 END) OVER w
        AS months_since_last_current,

    -- Carry forward vars
    current_actual_upb,
    loan_age,
    remaining_months_to_maturity

FROM performance_features

WINDOW w AS (
    PARTITION BY loan_id 
    ORDER BY reporting_period
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
);
""")